In [1]:
#IMPORT LIBRARIES

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib  # for saving/loading model and encoders
import gradio as gr
from google.colab import files

In [2]:
#UPLOAD DATASET

print("Please upload your CSV dataset (mental_health_dataset.csv)")

uploaded = files.upload()
file_name = list(uploaded.keys())[0]

# Load CSV
df = pd.read_csv(file_name)
print("✅ Dataset Loaded Successfully")
print(df.head())

Please upload your CSV dataset (mental_health_dataset.csv)


Saving mental_health_dataset.csv to mental_health_dataset.csv
✅ Dataset Loaded Successfully
   Student_ID  Age  Gender   GPA  Stress_Level  Anxiety_Score  \
0           1   23   Other  2.52             5             20   
1           2   19    Male  2.74             5              3   
2           3   21  Female  3.53             5             11   
3           4   18    Male  2.04             4             15   
4           5   19   Other  2.87             1              2   

   Depression_Score                                  Daily_Reflections  \
0                 6  Onto foreign do environmental anyone every nea...   
1                 7  Party but others visit admit industry country ...   
2                24  Religious sure wait do chance decade according...   
3                14            A task effect entire coach join series.   
4                 4  Knowledge several camera wait week write quali...   

   Sleep_Hours  Steps_Per_Day Mood_Description  Sentiment_Score  \
0    

In [3]:
#DATA PREPROCESSING

# Columns to encode
categorical_cols = ['Gender', 'Mood_Description', 'Mental_Health_Status']
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

# Drop columns not used in ML
if 'Daily_Reflections' in df.columns:
    df.drop(columns=['Daily_Reflections'], inplace=True)

# Features & Target
X = df.drop(columns=['Mental_Health_Status', 'Student_ID'])
y = df['Mental_Health_Status']

print(" Preprocessing Complete")

 Preprocessing Complete


In [4]:
#TRAIN-TEST SPLIT

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("Train-test split done")

Train-test split done


In [5]:
#TRAIN RANDOM FOREST MODEL

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
print(" Random Forest Model Trained Successfully")

 Random Forest Model Trained Successfully


In [6]:
# CHECK MODEL ACCURACY

from sklearn.metrics import accuracy_score
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy * 100:.2f}%")

Model Accuracy: 100.00%


In [7]:
#SAVE MODEL & ENCODERS

joblib.dump(model, "mental_health_model.pkl")
joblib.dump(label_encoders, "label_encoders.pkl")
print(" Model saved as 'mental_health_model.pkl'")
print(" Label encoders saved as 'label_encoders.pkl'")

 Model saved as 'mental_health_model.pkl'
 Label encoders saved as 'label_encoders.pkl'


In [8]:
#QUESTIONNAIRE RULE-BASED SYSTEM

def combined_mental_health_academic_rule_system(user_input):
    """
    user_input: dict containing all questionnaire responses
        Example keys:
        - Stress_Level (0-10)
        - Sleep_Hours (float)
        - Mood_Level (0-10)
        - Exercise (0-3)
        - Academic_Responses: list of 12 responses (0-3)
    Returns: dict with overall status and recommendations
    """
    recommendations = []

    # ---- Mental Health Rules ----
    stress = user_input.get("Stress_Level", 0)
    sleep = user_input.get("Sleep_Hours", 8)
    mood = user_input.get("Mood_Level", 5)
    exercise = user_input.get("Exercise", 1)

    # Mental health risk evaluation based purely on rules
    if stress > 7 or mood < 4:
        mental_status = "High Risk"
        recommendations.append("Seek professional counseling or mental health support.")
        if sleep < 6:
            recommendations.append("Improve sleep routine and avoid late-night screen time.")
        if stress > 7:
            recommendations.append("Practice stress management techniques like meditation or breathing exercises.")
        if exercise < 1:
            recommendations.append("Engage in light physical activity to boost mood.")
    elif 4 <= stress <= 7 or 4 <= mood <= 6:
        mental_status = "Mild Risk"
        recommendations.append("Maintain a balanced lifestyle and monitor mental health.")
        recommendations.append("Engage in physical activity and social interaction.")
    else:
        mental_status = "Low Risk"
        recommendations.append("Your mental health appears stable.")
        recommendations.append("Continue healthy habits and self-care routines.")

    # ---- Academic Stress Rules ----
    academic_responses = user_input.get("Academic_Responses", [0]*12)
    total_score = sum(academic_responses)

    if 0 <= total_score <= 7:
        academic_level = "Minimal Stress"
        recommendations.append("Student appears to be managing academic life well.")
        recommendations.append("Continue proper sleep, balanced study routines, and social interaction.")
    elif 8 <= total_score <= 14:
        academic_level = "Mild Stress"
        recommendations.append("Student may be experiencing some academic pressure.")
        recommendations.append("Improve time management and take short study breaks.")
    elif 15 <= total_score <= 25:
        academic_level = "Moderate Stress"
        recommendations.append("Student may be experiencing noticeable stress.")
        recommendations.append("Practice relaxation techniques such as deep breathing or mindfulness.")
        recommendations.append("Break tasks into smaller steps to reduce overwhelm.")
    elif 26 <= total_score <= 36:
        academic_level = "High Stress"
        recommendations.append("Student may be experiencing high academic stress.")
        recommendations.append("Seek support from mentors, teachers, counselors, or trusted individuals.")
    else:
        academic_level = "Invalid Score"
        recommendations.append("Please provide valid academic questionnaire responses (0-3 per question).")

    # ---- Return consolidated result ----
    return {
        "Mental_Health_Status": mental_status,
        "Academic_Stress_Level": academic_level,
        "Recommendations": list(dict.fromkeys(recommendations))  # remove duplicates
    }


# -------- Example Usage --------
user_input_example = {
    "Stress_Level": 8,
    "Sleep_Hours": 5,
    "Mood_Level": 3,
    "Exercise": 0,
    "Academic_Responses": [3,2,2,1,2,3,2,2,1,2,1,2]
}

result = combined_mental_health_academic_rule_system(user_input_example)
for key, val in result.items():
    print(key, ":", val)

Mental_Health_Status : High Risk
Academic_Stress_Level : Moderate Stress
Recommendations : ['Seek professional counseling or mental health support.', 'Improve sleep routine and avoid late-night screen time.', 'Practice stress management techniques like meditation or breathing exercises.', 'Engage in light physical activity to boost mood.', 'Student may be experiencing noticeable stress.', 'Practice relaxation techniques such as deep breathing or mindfulness.', 'Break tasks into smaller steps to reduce overwhelm.']
